In [0]:
# --- 1. Konfiguration ---
volume_path = "/Volumes/workspace/default/volume"
source_data_path = f"{volume_path}/enrollments_raw"  # Zielordner für den Auto Loader

# --- 2. Aufräumen (Tabellen und alte Ordner löschen) ---
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA default")
spark.sql("DROP TABLE IF EXISTS students")
spark.sql("DROP TABLE IF EXISTS enrollments_bronze")
spark.sql("DROP TABLE IF EXISTS enrollments_silver")
spark.sql("DROP TABLE IF EXISTS daily_student_courses")
dbutils.fs.rm(source_data_path, recurse=True)
print("✅ Alte Tabellen und Quelldaten-Ordner gelöscht.")

# --- 3. Students-Stammdaten aus dem echten JSON erstellen ---
students_df = spark.read.json(f"{volume_path}/students.json")
students_df.write.mode("overwrite").saveAsTable("workspace.default.students")
print("✅ Stammdaten-Tabelle 'workspace.default.students' aus students.json erstellt.")

# --- 4. Enrollments-Rohdaten für den Auto Loader simulieren ---
enrollments_df = (
    spark.read.format("json")
    .option("inferSchema", "true")
    .load(f"{volume_path}/enrollments.json")
)

(enrollments_df.write.mode("overwrite").format("json").save(source_data_path))

print(f"✅ Quelldaten aus enrollments.json in '{source_data_path}' geschrieben.")
print(
    "\n🚀 Setup abgeschlossen. Die DLT-Pipeline kann jetzt fehlerfrei gestartet werden."
)

# --- 5. CDC-Updates simulieren ---
from pyspark.sql.functions import current_timestamp, lit

# Wir nehmen die eben erstellten Studenten und fügen Metadaten für CDC hinzu
students_updates_df = (
    spark.table("workspace.default.students")
    .withColumn("operation", lit("INSERT"))
    .withColumn("sequence_id", current_timestamp())
)

# Diese Tabelle dient als Streaming-Quelle für AUTO CDC / APPLY CHANGES
students_updates_df.write.mode("overwrite").saveAsTable(
    "workspace.default.students_updates"
)
print("✅ CDC-Quelle 'workspace.default.students_updates' wurde vorbereitet.")